In [24]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()
HEADERS = { "User-Agent": os.getenv('SEC_USER_AGENT') }

In [27]:
def get_filing_text(ticker: str, form_type: str = '10-K') -> dict:
    """
    Fetches the actual text content of a filing.
    Returns the text and metadata about the filing.
    """
    # step 1: Get the company's CIK
    ticker_url = "https://www.sec.gov/files/company_tickers.json"
    resp = requests.get(ticker_url, headers=HEADERS)
    print(f"Ticker URL Status: {resp.status_code}")
    print(f"Response length: {len(resp.text)}")
    if resp.status_code != 200:
        print(f"Response: {resp.text[:500]}")
        return None
    
    tickers = resp.json()
    cik = None
    for entry in tickers.values():
        if entry['ticker'] == ticker.upper():
            cik = str(entry["cik_str"]).zfill(10)
            company_name = entry['title']
            break
    
    print(f"Company: {company_name} | CIK: {cik}")

    # step 2: Get the list of filings for this company
    submission_url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    resp = requests.get(submission_url, headers=HEADERS)
    print(f"Submission URL Status: {resp.status_code}")
    if resp.status_code != 200:
        print(f"Response: {resp.text[:500]}")
        return None
    
    subs = resp.json()

        
    # Step 3: Find the most recent 10-K
    filings = subs["filings"]["recent"]
    forms = filings["form"]
    dates = filings["filingDate"]
    accessions = filings["accessionNumber"]
    primary_document = filings["primaryDocument"]

    # Find first matching form
    accession_idx = None
    for i, form in enumerate(forms):
        if form == form_type:
            accession_idx = i
            filing_date = dates[i]
            doc_10_k = primary_document[i]
            print(f"Found {form_type} filed on: {filing_date}")
            break
    
    accession_with_dashes = accessions[accession_idx]
    accession_no_dashes = accession_with_dashes.replace("-", "")

    # Step 4: Construct the direct document URL
    # Try common naming patterns for 10-K documents
    doc_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession_no_dashes}/{doc_10_k}"
    print(f"Trying document URL: {doc_url}")
    
    doc_resp = requests.get(doc_url, headers=HEADERS)
    print(f"Document URL Status: {doc_resp.status_code}")
    
    if doc_resp.status_code == 404:
        # Try alternative naming
        doc_url = f"https://www.sec.gov/cgi-bin/viewer?action=view&cik={cik}&accession_number={accession_with_dashes}&xbrl_type=v"
        print(f"Trying alternative URL: {doc_url}")
        doc_resp = requests.get(doc_url, headers=HEADERS)
        print(f"Status: {doc_resp.status_code}")
    
    return {
        "ticker": ticker,
        "company": company_name,
        "form_type": form_type,
        "filing_date": filing_date,
        "raw_html": doc_resp.text,
        "url": doc_url
    }



In [28]:
filing = get_filing_text("AAPL")
print(f"\nFetched {len(filing['raw_html']):,} characters of HTML")

Ticker URL Status: 200
Response length: 795117
Company: Apple Inc. | CIK: 0000320193
Submission URL Status: 200
Found 10-K filed on: 2025-10-31
Trying document URL: https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm
Document URL Status: 200

Fetched 1,520,208 characters of HTML


In [29]:
from bs4 import BeautifulSoup
import re

# Key concept: 10-K filings have standard sections defined by the SEC
# We want to identify and label each section so our chunks carry context
SEC_SECTIONS = {
    "item_1":  "Business Overview",
    "item_1a": "Risk Factors",
    "item_1b": "Unresolved Staff Comments",
    "item_1c": "Cybersecurity",
    "item_7":  "Management Discussion and Analysis",
    "item_7a": "Quantitative Disclosures About Market Risk",
    "item_8":  "Financial Statements",
}

In [30]:
def clean_and_section(raw_html: str) -> list[dict]:
    """
    Parses HTML, removes boilerplate, and splits into labeled sections.
    Returns a list of {section_name, text} dicts.
    """
    # Parse HTML
    soup = BeautifulSoup(raw_html, "lxml")
    
    # Remove scripts, styles, tables of contents
    for tag in soup(["script", "style", "head"]):
        tag.decompose()
    
    # Get raw text
    full_text = soup.get_text(separator="\n")
    
    # Clean up whitespace — this is a common data cleaning pattern
    lines = [line.strip() for line in full_text.splitlines()]
    lines = [line for line in lines if len(line) > 20]  # drop short noise lines
    full_text = "\n".join(lines)
    
    # Find section boundaries using regex
    # 10-K filings always label sections as "Item 1A.", "ITEM 1A.", etc.
    section_pattern = re.compile(
        r'(ITEM\s+\d+[A-Z]?\.|Item\s+\d+[A-Z]?\.)',
        re.IGNORECASE
    )
    
    # Split text at each "Item X." marker
    parts = section_pattern.split(full_text)
    
    sections = []
    current_label = "preamble"
    
    for i, part in enumerate(parts):
        if section_pattern.match(part.strip()):
            current_label = part.strip().lower().replace(" ", "_").replace(".", "")
        else:
            if len(part.strip()) > 100:  # skip tiny fragments
                sections.append({
                    "section": current_label,
                    "text": part.strip()
                })
    
    print(f"Extracted {len(sections)} sections")
    for s in sections[:8]:
        print(f"  [{s['section']}]: {len(s['text']):,} chars")
    
    return sections

In [31]:
sections = clean_and_section(filing["raw_html"])

/tmp/ipykernel_121971/829934400.py:7: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(raw_html, "lxml")


Extracted 18 sections
  [preamble]: 31,437 chars
  [item_1a]: 67,986 chars
  [item_1c]: 2,690 chars
  [item_2]: 508 chars
  [item_3]: 5,379 chars
  [item_5]: 2,249 chars
  [item_7]: 16,568 chars
  [item_7a]: 2,857 chars


In [32]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

In [33]:
# Why chunk? LLMs have context limits. A 10-K has ~150K words.
# We can only send ~2K words per query. So we pre-split and retrieve
# only the RELEVANT chunks. This is the core idea behind RAG.

def chunk_sections(sections: list[dict], ticker: str, filing_date: str) -> list[dict]:
    """
    Splits each section into overlapping chunks.
    
    chunk_size=500: ~500 tokens per chunk (roughly 375 words)
    chunk_overlap=50: last 50 tokens of chunk N = first 50 tokens of chunk N+1
                      This prevents losing context at boundaries.
    """

    #tiktoken counts tokens the same way claude does
    tokenizer = tiktoken.get_encoding('cl100k_base')

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        length_function=lambda text: len(tokenizer.encode(text)),
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    all_chunks = []

    for section in sections:
        # split this section's text into chunks
        texts = splitter.split_text(section['text'])

        for i, chunk_text in enumerate(texts):
            all_chunks.append({
                "ticker": ticker,
                "filing_date": filing_date,
                "section": section["section"],
                "chunk_index": i,
                "text": chunk_text,
                "token_count": len(tokenizer.encode(chunk_text))
            })
    
    print(f"Total chunks: {len(all_chunks)}")
    print(f"Avg tokens per chunk: {sum(c['token_count'] for c in all_chunks) / len(all_chunks):.0f}")
    sample_idx = min(10, len(all_chunks) - 1)
    print(f"Sample chunk:\n---\n{all_chunks[sample_idx]['text'][:300]}\n---")
    
    return all_chunks

In [34]:
chunks = chunk_sections(sections, filing["ticker"], filing["filing_date"])

Total chunks: 103
Avg tokens per chunk: 394
Sample chunk:
---
The Company’s cloud services store and keep customers’ content up-to-date and available across multiple Apple devices and Windows personal computers.
The Company operates various platforms, including the App Store
that allow customers to discover and download applications and digital content, such a
---


In [35]:
from sentence_transformers import SentenceTransformer

In [36]:
# BGE (Beijing Academy of AI General Embeddings) outperforms OpenAI
# text-embedding-3-small on financial text. It runs locally — free.
# "Embedding" = converting text into a list of numbers that captures meaning.
# Similar sentences → similar number lists → close together in vector space.

print("Loading BGE model (downloads ~270MB on first run)...")
model = SentenceTransformer("BAAI/bge-base-en-v1.5")

Loading BGE model (downloads ~270MB on first run)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [37]:
def embed_chunks(chunks: list[dict], batch_size: int = 32) -> list[dict]:
    """
    Adds an 'embedding' field to each chunk.
    Processes in batches for memory efficiency.
    """
    texts = [c["text"] for c in chunks]

    print(f"Embedding {len(texts)} chunks in batches of {batch_size}...")

    # encode() returns a numpy array of shape (num_chunks, 768)
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True  # normalizing improves cosine similarity search
    )

    # Add embedding to each chunk dict
    for chunk, embedding in zip(chunks, embeddings):
        chunk["embedding"] = embedding.tolist()  # convert numpy → plain list

    print(f"Done. Each embedding has {len(chunks[0]['embedding'])} dimensions.")
    return chunks

In [38]:
chunks_with_embeddings = embed_chunks(chunks)

Embedding 103 chunks in batches of 32...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Done. Each embedding has 768 dimensions.


In [39]:
import psycopg2
from psycopg2.extras import execute_values
import os

In [51]:
def setup_database(conn_str: str):
    """
    Creates the table if it doesn't exist.
    This is the schema for your entire RAG knowledge base.
    """
    conn = psycopg2.connect(conn_str)
    cur = conn.cursor()

    # Enable the pgvector extension
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

    # Create our chunks table
    # vector(768) = a column that stores 768-dimensional vectors
    cur.execute("""
        CREATE TABLE IF NOT EXISTS filing_chunks (
            id          SERIAL PRIMARY KEY,
            ticker      TEXT NOT NULL,
            filing_date TEXT NOT NULL,
            section     TEXT NOT NULL,
            chunk_index INTEGER NOT NULL,
            text        TEXT NOT NULL,
            token_count INTEGER,
            embedding   vector(768),
            created_at  TIMESTAMP DEFAULT NOW()
        );
    """)

    # Create an HNSW index for fast approximate nearest-neighbor search
    # This makes similarity search fast even with millions of chunks
    cur.execute("""
        CREATE INDEX IF NOT EXISTS chunks_embedding_idx
        ON filing_chunks
        USING hnsw (embedding vector_cosine_ops);
    """)

    conn.commit()
    cur.close()
    conn.close()
    print("Database and table ready.")

In [42]:
def store_chunks(chunks: list[dict], conn_str: str):
    """Bulk-inserts all chunks into the database."""
    conn = psycopg2.connect(conn_str)
    cur = conn.cursor()

    # Prepare rows for bulk insert
    rows = [
        (
            c["ticker"],
            c["filing_date"],
            c["section"],
            c["chunk_index"],
            c["text"],
            c["token_count"],
            c["embedding"]
        )
        for c in chunks
    ]

    execute_values(
        cur,
        """INSERT INTO filing_chunks
           (ticker, filing_date, section, chunk_index, text, token_count, embedding)
           VALUES %s""",
        rows,
        template="(%s, %s, %s, %s, %s, %s, %s::vector)"
    )

    conn.commit()
    print(f"Stored {len(rows)} chunks in database.")
    cur.close()
    conn.close()

In [52]:
# Run it
CONN_STR = os.getenv("DATABASE_URL")
setup_database(CONN_STR)
# store_chunks(chunks_with_embeddings, CONN_STR)

Database and table ready.


In [44]:
def search_similar(query: str, ticker: str, top_k: int = 5) -> list[dict]:
    """
    The core of RAG: convert query to embedding, find similar chunks.
    This is what replaces keyword search with semantic understanding.
    """

    # Embed the query using the same model
    query_embedding = model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    conn = psycopg2.connect(CONN_STR)
    cur = conn.cursor()

    # pgvector's <=> operator = cosine distance (lower = more similar)
    # 1 - cosine_distance = cosine similarity (higher = more similar)
    cur.execute("""
        SELECT
            section,
            chunk_index,
            text,
            1 - (embedding <=> %s::vector) AS similarity
        FROM filing_chunks
        WHERE ticker = %s
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
    """, (query_embedding, ticker, query_embedding, top_k))

    results = []
    for row in cur.fetchall():
        results.append({
            "section": row[0],
            "chunk_index": row[1],
            "text": row[2],
            "similarity": round(row[3], 4)
        })

    cur.close()
    conn.close()
    return results

In [47]:
query = "What are Apple's main cybersecurity and data privacy risks?"
results = search_similar(query, "AAPL", top_k=3)

print(f"Query: '{query}'\n")
for i, r in enumerate(results):
    print(f"Result {i+1} | Section: {r['section']} | Similarity: {r['similarity']}")
    print(f"{r['text'][:400]}")
    print("---")

Query: 'What are Apple's main cybersecurity and data privacy risks?'

Result 1 | Section: item_1a | Similarity: 0.704
Apple Inc. | 2025 Form 10-K | 11
As with all companies, the security the Company has implemented may not be sufficient for all eventualities and are vulnerable to hacking, ransomware attacks, employee error, malfeasance, system error, faulty password management or other irregularities. For example, third parties can fraudulently induce the Company’s or its suppliers’ and other third parties’ emplo
---
Result 2 | Section: item_1a | Similarity: 0.7025
The Company’s business also requires it to share confidential information with suppliers and other third parties. The Company relies on global suppliers that are also exposed to ransomware and other malicious attacks that can disrupt business operations. Although the Company takes steps to secure confidential information that is provided to or accessible by third parties working on the Company’s b
---
Result 3 | Section: ite

In [ ]:
# Ingest all 5 companies — takes ~10 minutes total
for ticker in ["AAPL", "MSFT", "GOOGL", "META", "NVDA"]:
    print(f"\n{'='*40}")
    print(f"Processing {ticker}...")
    filing = get_filing_text(ticker)
    sections = clean_and_section(filing["raw_html"])
    chunks = chunk_sections(sections, ticker, filing["filing_date"])
    chunks = embed_chunks(chunks)
    store_chunks(chunks, CONN_STR)
    print(f"✓ {ticker} done")


Processing AAPL...
Ticker URL Status: 200
Response length: 795117
Company: Apple Inc. | CIK: 0000320193
Submission URL Status: 200
Found 10-K filed on: 2025-10-31
Trying document URL: https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm
Document URL Status: 200


/tmp/ipykernel_121971/829934400.py:7: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(raw_html, "lxml")


Extracted 18 sections
  [preamble]: 31,437 chars
  [item_1a]: 67,986 chars
  [item_1c]: 2,690 chars
  [item_2]: 508 chars
  [item_3]: 5,379 chars
  [item_5]: 2,249 chars
  [item_7]: 16,568 chars
  [item_7a]: 2,857 chars
Total chunks: 103
Avg tokens per chunk: 394
Sample chunk:
---
The Company’s cloud services store and keep customers’ content up-to-date and available across multiple Apple devices and Windows personal computers.
The Company operates various platforms, including the App Store
that allow customers to discover and download applications and digital content, such a
---
Embedding 103 chunks in batches of 32...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Done. Each embedding has 768 dimensions.
Stored 103 chunks in database.
✓ AAPL done

Processing MSFT...
Ticker URL Status: 200
Response length: 795117
Company: MICROSOFT CORP | CIK: 0000789019
Submission URL Status: 200
Found 10-K filed on: 2026-07-29
Trying document URL: https://www.sec.gov/Archives/edgar/data/789019/000119312526323660/msft-20260630.htm
Document URL Status: 200
Extracted 12 sections
  [preamble]: 157,711 chars
  [item_5]: 1,196 chars
  [item_7]: 48,521 chars
  [item_7a]: 1,645 chars
  [item_8]: 82,195 chars
  [item_9]: 2,628 chars
  [item_9a]: 5,213 chars
  [item_10]: 1,884 chars
Total chunks: 147
Avg tokens per chunk: 427
Sample chunk:
---
us-gaap:FairValueInputsLevel2Member
us-gaap:CorporateDebtSecuritiesMember
us-gaap:RevenueCommissionersIrelandMember
msft:MorePersonalComputingMember
msft:IssuanceOfLongTermDebtTenMember
msft:IssuanceOfLongTermDebtThreeMember
msft:IssuanceOfLongTermDebtElevenMember
msft:OperatingLeaseMember
msft:Shar
---
Embedding 147 chunks in batc

...

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Done. Each embedding has 768 dimensions.
Stored 157 chunks in database.
✓ GOOGL done

Processing META...
Ticker URL Status: 200
Response length: 795117
Company: Meta Platforms, Inc. | CIK: 0001326801
Submission URL Status: 200
Found 10-K filed on: 2026-01-29
Trying document URL: https://www.sec.gov/Archives/edgar/data/1326801/000162828026003942/meta-20251231.htm
Document URL Status: 200
Extracted 2 sections
  [preamble]: 494,341 chars
  [item 5]: 8,739 chars
Total chunks: 240
Avg tokens per chunk: 412
Sample chunk:
---
Security Ownership of Certain Beneficial Owners and Management and Related Stockholder Matters
Certain Relationships and Related Transactions, and Director Independence
Principal Accountant Fees and Services
Exhibit and Financial Statement Schedules
NOTE ABOUT FORWARD-LOOKING STATEMENTS
This Annual 
---
Embedding 240 chunks in batches of 32...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Done. Each embedding has 768 dimensions.
Stored 240 chunks in database.
✓ META done

Processing NVDA...
Ticker URL Status: 200
Response length: 795117
Company: NVIDIA CORP | CIK: 0001045810
Submission URL Status: 200
Found 10-K filed on: 2026-02-25
Trying document URL: https://www.sec.gov/Archives/edgar/data/1045810/000104581026000021/nvda-20260125.htm
Document URL Status: 200
Extracted 25 sections
  [preamble]: 61,507 chars
  [item_1a]: 1,138 chars
  [item_1a]: 6,173 chars
  [item_1a]: 114,271 chars
  [item_1c]: 2,051 chars
  [item_1a]: 2,754 chars
  [item_3]: 188 chars
  [item_5]: 4,811 chars
Total chunks: 165
Avg tokens per chunk: 393
Sample chunk:
---
Unresolved Staff Comments
Mine Safety Disclosures
Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities
Management’s Discussion and Analysis of Financial Condition and Results of Operations
Quantitative and Qualitative Disclosures About Market R
---
Embedding 165 chunks in batches

Batches:   0%|          | 0/6 [00:00<?, ?it/s]